# 🧹 Cleanup Workshop Resources

Use this notebook to **delete all Azure resources** created for the RAG Workshop.

## ⚠️ Warning
**This action is irreversible.** It will permanently delete:
- The Resource Group (`rg-rag-workshop`)
- All data in Azure Storage
- All Search Indexes
- The deployed AI Models
- The Foundry Hub & Project

In [ ]:
import subprocess
import json

# Configuration
RESOURCE_GROUP = "rg-rag-workshop"
LOCATION = "swedencentral"

print(f"🗑️ Preparing to delete resource group: {RESOURCE_GROUP}")

In [ ]:
# Step 1: Login Check
try:
    result = subprocess.run(["az", "account", "show"], capture_output=True, text=True)
    if result.returncode == 0:
        print("✅ Logged in")
    else:
        print("❌ Not logged in. Please run 'az login' in a terminal.")
        raise Exception("Not logged in")
except FileNotFoundError:
    print("❌ Azure CLI not found")
    raise

In [ ]:
# Step 2: Delete Resource Group
print(f"🔥 Deleting Resource Group '{RESOURCE_GROUP}'... (This takes 5-10 minutes)")

result = subprocess.run(
    ["az", "group", "delete", "--name", RESOURCE_GROUP, "--yes", "--no-wait"],
    capture_output=True, 
    text=True
)

if result.returncode == 0:
    print("✅ Resource Group deletion started in background.")
else:
    print(f"❌ Failed to delete RG: {result.stderr}")

In [ ]:
# Step 3: Purge Soft-Deleted Resources
# We need to purge the Cognitive Services to allow immediate re-creation with the same name.

print("🧹 Checking for soft-deleted resources to purge...")

# Get list of deleted accounts in the region
list_result = subprocess.run(
    ["az", "cognitiveservices", "account", "list-deleted", "--output", "json"],
    capture_output=True,
    text=True
)

try:
    deleted_accounts = json.loads(list_result.stdout)
    purged_count = 0
    
    for account in deleted_accounts:
        if account['location'] == LOCATION:
            name = account['name']
            # Only purge if it looks like one of ours (starts with ai-ragworkshop, etc)
            if 'ragworkshop' in name:
                print(f"   Purging: {name}...")
                subprocess.run(
                    ["az", "cognitiveservices", "account", "purge", 
                     "--location", LOCATION, "--name", name, "--resource-group", RESOURCE_GROUP],
                    capture_output=True
                )
                purged_count += 1
    
    if purged_count == 0:
        print("   No matching soft-deleted resources found.")
    else:
        print(f"✅ Purged {purged_count} resources.")

except json.JSONDecodeError:
    print("⚠️ Could not parse deleted resources list.")

In [ ]:

# Step 4: Delete .env file
from pathlib import Path
import os

print("🗑️ Removing .env file...")
env_path = Path("../../.env").resolve()

if env_path.exists():
    try:
        os.remove(env_path)
        print(f"✅ Deleted: {env_path}")
    except Exception as e:
        print(f"❌ Failed to delete .env: {e}")
else:
    print(f"ℹ️ {env_path} does not exist.")



## 🔄 Ready to Re-Deploy?

Once the deletion is complete, you can go back to `setup.ipynb` and run **Step 3** to recreate the environment from scratch.

The new deployment will create:
1. **Unified AI Services** (Kind: `AIServices`)
   - This single resource provides GPT-4o, Document Intelligence, and Content Understanding.
   - It is fully compatible with the **Azure AI Foundry Portal** for model testing and development.
2. **Azure AI Search**
3. **Storage Account**

*Note: We are not deploying the "Hub & Project" resources to avoid network restriction complexity.*
